# metrics_spark.py 逐步驗證 notebook

逐步檢查 `src/recsys_tfb/evaluation/metrics_spark.py` 每個 function 的中間結果（first-principles redesign 版本）。

**資料來源**：`recsys_tfb.io.extract.extract_Xy_with_groups` 從本地 cache 取 test 集 → `model.predict` → 拼成 `(snap_date, cust_id, prod_name, label, score)` Spark DataFrame 餵入 metrics_spark。

對照 module docstring 的 layered structure：

```
Layer 1 row-level     : rank_within_query / add_query_total_rel / add_row_contributions
Layer 2 per-query     : compute_per_query_metrics
Layer 3 aggregations  : aggregate_overall / aggregate_per_segment
                       / aggregate_per_item / macro_average
Layer 4 orchestrator  : compute_all_metrics
```

## 0.1 環境 + parameters

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:.6f}".rstrip("0").rstrip("."))
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
print("cwd:", Path.cwd())

In [ ]:
import json

from recsys_tfb.core.config import ConfigLoader
from recsys_tfb.core.versioning import resolve_base_dataset_version, resolve_model_version

config = ConfigLoader("conf", env="local")
data_dir = Path("data")

dataset_version = resolve_base_dataset_version(data_dir / "dataset", None)
model_version = resolve_model_version(data_dir / "models", None)

parameters = config.get_parameters()
parameters["base_dataset_version"] = dataset_version
parameters["model_version"] = model_version

print(f"dataset_version = {dataset_version}")
print(f"model_version   = {model_version}")
print("evaluation:", parameters.get("evaluation"))

## 0.2 載入 preprocessor + model + test parquet handle

In [ ]:
from recsys_tfb.io.handles import ParquetHandle
from recsys_tfb.io.model_adapter_dataset import ModelAdapterDataset

preprocessor_path = data_dir / "dataset" / dataset_version / "preprocessor.json"
with open(preprocessor_path) as f:
    preprocessor_metadata = json.load(f)
print("preprocessor keys:", list(preprocessor_metadata.keys()))
print("feature_columns count:", len(preprocessor_metadata["feature_columns"]))
print("categorical_columns:", preprocessor_metadata["categorical_columns"])

model_path = data_dir / "models" / "best" / "model.txt"
model = ModelAdapterDataset(filepath=str(model_path)).load()
print("model:", type(model).__name__)

test_cache_path = data_dir / "recsys_cache" / dataset_version / "test_model_input.parquet"
assert test_cache_path.exists(), f"找不到 test cache：{test_cache_path}"
test_handle = ParquetHandle(path=str(test_cache_path))
print("test_handle.path =", test_handle.path)

## 0.3 `extract_Xy_with_groups` → 預測分數

In [ ]:
from recsys_tfb.io.extract import extract_Xy_with_groups
from recsys_tfb.core.schema import get_schema

schema = get_schema(parameters)
time_col = schema["time"]
entity_cols = schema["entity"]
item_col = schema["item"]
label_col = schema["label"]
score_col = schema["score"]
group_cols = [time_col] + entity_cols
print("schema:", schema)
print("group_cols:", group_cols)

X, y, groups = extract_Xy_with_groups(
    test_handle,
    preprocessor_metadata,
    parameters,
    filter_groups_with_positives=True,
)
scores = model.predict(X)
print(f"X.shape={X.shape}  y.sum()={int(y.sum())}  n_groups={int(groups.max()) + 1}  scores.shape={scores.shape}")

## 0.4 組裝供 `metrics_spark` 使用的 Spark DataFrame

`extract_Xy_with_groups` 回傳 numpy，沒有 identity 欄位，這邊把同一 parquet 用同樣的 filter 再讀一次取 `(snap_date, cust_id, prod_name, label)`，把 scores 接回去。

In [ ]:
pdf_all = test_handle.to_pandas()
group_pos = pdf_all.groupby(group_cols, sort=False)[label_col].transform("sum")
pdf_pos = pdf_all.loc[group_pos > 0].reset_index(drop=True)
assert len(pdf_pos) == len(scores), (
    f"row 數對不上：pdf_pos={len(pdf_pos)} scores={len(scores)}"
)

keep_cols = [time_col] + entity_cols + [item_col, label_col]
# test cache 沒有 cust_segment_typ；挑一個既有的 customer-level 屬性作為 segment 示範
seg_candidates = [c for c in ["cust_segment_typ", "risk_attr", "income_level", "marital_status"] if c in pdf_pos.columns]
demo_seg_col = seg_candidates[0] if seg_candidates else None
if demo_seg_col:
    keep_cols.append(demo_seg_col)

eval_pdf = pdf_pos[keep_cols].copy()
eval_pdf[score_col] = scores
print(f"eval_pdf shape={eval_pdf.shape}  demo_seg_col={demo_seg_col}")
display(eval_pdf.head())

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("verify_metrics_spark")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

eval_predictions = spark.createDataFrame(
    eval_pdf.assign(**{time_col: eval_pdf[time_col].astype(str)})
)
eval_predictions.printSchema()
print("row count:", eval_predictions.count())
eval_predictions.show(5, truncate=False)

## 0.5 解析 k_values

In [ ]:
from recsys_tfb.evaluation.metrics_spark import _resolve_k_values

eval_params = parameters.get("evaluation", {})
k_values_raw = eval_params.get("k_values", [5, "all"])
segment_columns = eval_params.get("segment_columns", [])
n_products = eval_predictions.select(item_col).distinct().count()
k_values = _resolve_k_values(k_values_raw, n_products)

print(f"k_values_raw = {k_values_raw}")
print(f"n_products   = {n_products}")
print(f"k_values     = {k_values}  (\"all\" 解析為 {n_products})")
print(f"segment_columns (parameters_evaluation.yaml) = {segment_columns}")

## A1. `rank_within_query` — 1-based pos within query

In [ ]:
from recsys_tfb.evaluation import metrics_spark as ms
from pyspark.sql import functions as F

ranked = ms.rank_within_query(eval_predictions, group_cols, score_col)
print("schema after rank_within_query:")
ranked.printSchema()

sample_cust = ranked.select(*entity_cols).distinct().limit(1).collect()[0]
filt = ranked
for c in entity_cols:
    filt = filt.filter(F.col(c) == sample_cust[c])
filt.orderBy("pos").show(50, truncate=False)

In [ ]:
# Sanity：每個 query pos 從 1 連到 n_rows
check = (
    ranked.groupBy(*group_cols)
    .agg(
        F.min("pos").alias("min_pos"),
        F.max("pos").alias("max_pos"),
        F.count("*").alias("n_rows"),
    )
)
check.show(10)
bad = check.filter((F.col("min_pos") != 1) | (F.col("max_pos") != F.col("n_rows")))
print(f"queries with broken pos numbering: {bad.count()}  (預期 0)")

## A2. `add_query_total_rel` — `total_rel = sum(label)` per query

Window 把每個 query 的正樣本總數寫進每列；後面 caller 會 `filter(total_rel > 0)`。

In [ ]:
with_total = ms.add_query_total_rel(ranked, group_cols, label_col)
with_total.printSchema()

with_total.select(*group_cols, label_col, "pos", "total_rel") \
    .orderBy(*group_cols, "pos").limit(20).show(truncate=False)

with_total.groupBy("total_rel").count().orderBy("total_rel").show()
n_excluded = with_total.filter(F.col("total_rel") == 0).select(*group_cols).distinct().count()
print(f"queries with total_rel == 0 (將被過濾): {n_excluded}")

In [ ]:
df_with_pos = with_total.filter(F.col("total_rel") > 0)
print(f"rows after filter total_rel > 0: {df_with_pos.count()}")

## A3. `add_row_contributions`

每列加上 `cum_rel`、`prec_at_pos`、`dcg_term`、以及對每個 K 的 `top_k@K`、`ap_contrib@K`、`ndcg_contrib@K`。

In [ ]:
enriched = ms.add_row_contributions(df_with_pos, group_cols, label_col, k_values)
print("contributions schema:")
enriched.printSchema()
print("\nk_values:", k_values)

In [ ]:
pos_query = (
    enriched.filter(F.col("total_rel") > 0)
    .select(*entity_cols, "total_rel").distinct().limit(1).collect()[0]
)
filt = enriched
for c in entity_cols:
    filt = filt.filter(F.col(c) == pos_query[c])
show_cols = [time_col, *entity_cols, item_col, label_col, "pos", "total_rel",
             "cum_rel", "prec_at_pos", "dcg_term"]
for k in k_values:
    show_cols += [f"top_k@{k}", f"ap_contrib@{k}", f"ndcg_contrib@{k}"]
filt.orderBy("pos").select(*show_cols).show(50, truncate=False)

In [ ]:
import math

rows = filt.orderBy("pos").collect()
labels = [r[label_col] for r in rows]
positions = [r["pos"] for r in rows]
cum_rel_expected, running = [], 0
for lb in labels:
    running += lb
    cum_rel_expected.append(running)

assert [r["cum_rel"] for r in rows] == cum_rel_expected
for r in rows:
    assert abs(r["prec_at_pos"] - r["cum_rel"] / r["pos"]) < 1e-12
    assert abs(r["dcg_term"] - r[label_col] / math.log2(r["pos"] + 1)) < 1e-12

for k in k_values:
    for r in rows:
        assert r[f"top_k@{k}"] == (1.0 if r["pos"] <= k else 0.0)

for k in k_values:
    for r in rows:
        expected = r["prec_at_pos"] * r[label_col] * r[f"top_k@{k}"]
        assert abs(r[f"ap_contrib@{k}"] - expected) < 1e-12

total_rel = rows[0]["total_rel"]
for k in k_values:
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, min(total_rel, k) + 1))
    for r in rows:
        expected = (r["dcg_term"] * r[f"top_k@{k}"] / idcg) if idcg > 0 else 0.0
        assert abs(r[f"ndcg_contrib@{k}"] - expected) < 1e-9, f"ndcg_contrib@{k} 失準"

print("A3 row-level contribution 全部通過手算對齊。")

## Layer 2. `compute_per_query_metrics` — 一個 query 一列

把 row-level contribution 聚合成 query 層級 metric。輸出每個 K 都有 `map@K`、`ndcg@K`、`precision@K`、`recall@K`。

> 注意 K=n_products 時：`precision@K` 退化成該 query 的 base rate `total_rel/n_products`；`recall@K` 恆等於 1.0（保留作 sanity check，但語意上是 degenerate）。

In [ ]:
carry = [demo_seg_col] if demo_seg_col else []
per_query = ms.compute_per_query_metrics(
    enriched, group_cols, label_col, k_values, carry_cols=carry
)
per_query.printSchema()
per_query.orderBy(*group_cols).limit(10).show(truncate=False)

In [ ]:
# Sanity：每個 query 一列
distinct_queries = per_query.select(*group_cols).distinct().count()
total_rows = per_query.count()
print(f"distinct queries = {distinct_queries}, total rows = {total_rows}  (應相等)")
assert distinct_queries == total_rows

## B1. `aggregate_overall` — query 等權平均 per-query metrics

吃 Layer 2 的 `per_query` DataFrame，對每個 metric 跨 query 取 mean。

In [ ]:
overall = ms.aggregate_overall(per_query, k_values)
print("overall =")
for k_name, v in overall.items():
    print(f"  {k_name}: {v:.6f}")

In [ ]:
# Sanity：自己對 per_query 各欄位取 mean，應與 overall 相等
metric_cols = []
for k in k_values:
    metric_cols += [f"map@{k}", f"ndcg@{k}", f"precision@{k}", f"recall@{k}"]

manual = per_query.agg(*[F.mean(c).alias(c) for c in metric_cols]).collect()[0].asDict()
diffs = []
for c in metric_cols:
    if abs(overall[c] - manual[c]) > 1e-12:
        diffs.append((c, overall[c], manual[c]))
print("overall vs 手算 per_query.mean() 差異 (預期空):", diffs)

In [ ]:
# 觀察 K=n_products 時 precision/recall 的退化行為
k_all = max(k_values)
if k_all == n_products:
    print(f"K = n_products = {k_all}：")
    print(f"  precision@{k_all} = {overall[f'precision@{k_all}']:.6f}  (= base rate)")
    print(f"  recall@{k_all}    = {overall[f'recall@{k_all}']:.6f}  (恆等於 1.0)")
    # 手算 base rate
    n_pos_total = per_query.agg(F.sum("total_rel")).collect()[0][0]
    n_queries = per_query.count()
    expected_prec = (n_pos_total / n_queries) / n_products
    print(f"  expected base rate = (Σ total_rel / n_queries) / n_products = {expected_prec:.6f}")

## B2. `aggregate_per_item` — 按 item 看 attribution metric

不再叫 `aggregate_by_row_dimension`，且**不再輸出 `precision@K` / `recall@K`**（per-item 語意不對等）。

新輸出（per item）：

| metric | 公式（label=1 rows for P） | 直白語意 |
| --- | --- | --- |
| `hit_rate@K` | `mean(top_k@K)` | P 真的是正解時，被排進 top-K 的機率（item-level marginal recall） |
| `map_attr@K` | `mean(ap_contrib@K)` | per-row AP@K 貢獻平均 |
| `ndcg_attr@K` | `mean(ndcg_contrib@K)` | per-row nDCG@K 貢獻平均（已 iDCG 標準化）|
| `mean_pos` | `mean(pos)` | P 作為正解時的平均排名位置 |

In [ ]:
per_item = ms.aggregate_per_item(enriched, [item_col], label_col, k_values)
per_item_df = pd.DataFrame(per_item).T.sort_index()
display(per_item_df)

In [ ]:
# Sanity：手動重算 hit_rate@K / map_attr@K / ndcg_attr@K / mean_pos
rel_only = enriched.filter(F.col(label_col) == 1)
manual_aggs = [F.mean(F.col("pos").cast("double")).alias("mean_pos")]
for k in k_values:
    manual_aggs += [
        F.mean(f"top_k@{k}").alias(f"hit_rate@{k}"),
        F.mean(f"ap_contrib@{k}").alias(f"map_attr@{k}"),
        F.mean(f"ndcg_contrib@{k}").alias(f"ndcg_attr@{k}"),
    ]
manual_per_item = (
    rel_only.groupBy(item_col).agg(*manual_aggs).orderBy(item_col)
)
manual_per_item.show(truncate=False)

## B3. `aggregate_per_segment` — 按 segment 看 per-query metric（每客戶等權）

這次接 `per_query`（Layer 2 的中間表），不再像舊版自己重算一次 per-query 公式 —— 把 per_query DataFrame 餵進來、按 seg 分組取 mean 就好。

注意 `cust_segment_typ`（parameters_evaluation 預設）在 test cache 不存在，這裡用 `demo_seg_col` 示範。

In [ ]:
if demo_seg_col is None:
    print("test cache 沒有 segment 候選欄位，跳過 B3")
    per_segment = {}
else:
    per_segment = ms.aggregate_per_segment(per_query, demo_seg_col, k_values)
    per_segment_df = pd.DataFrame(per_segment).T.sort_index()
    display(per_segment_df)

In [ ]:
# Sanity：手動 per_query.groupBy(seg).mean(...) 對照
if demo_seg_col is not None:
    manual_per_seg = (
        per_query.groupBy(demo_seg_col)
        .agg(*[F.mean(c).alias(c) for c in metric_cols])
        .orderBy(demo_seg_col)
    )
    manual_per_seg.show(truncate=False)

## C. `macro_average` — 跨 dim key 取平均

In [ ]:
macro_avg = {"by_item": ms.macro_average(per_item)}
if per_segment:
    macro_avg["by_segment"] = ms.macro_average(per_segment)

print("macro_avg.by_item:")
for k_name, v in macro_avg["by_item"].items():
    print(f"  {k_name}: {v:.6f}")

if "by_segment" in macro_avg:
    print("\nmacro_avg.by_segment:")
    for k_name, v in macro_avg["by_segment"].items():
        print(f"  {k_name}: {v:.6f}")

## D. `compute_all_metrics` 端到端對照

In [ ]:
result = ms.compute_all_metrics(eval_predictions, parameters)
print("keys:", list(result.keys()))
print(f"n_queries          = {result['n_queries']}")
print(f"n_excluded_queries = {result['n_excluded_queries']}")

print("\nresult['overall']:")
for k_name, v in result["overall"].items():
    print(f"  {k_name}: {v:.6f}")

mismatch = []
for k_name, v in overall.items():
    if abs(result["overall"].get(k_name, float('nan')) - v) > 1e-12:
        mismatch.append((k_name, v, result["overall"].get(k_name)))
print("\noverall 與逐步驗證差異 (預期空):", mismatch)

In [ ]:
print("per_item (compute_all_metrics):")
display(pd.DataFrame(result["per_item"]).T.sort_index())

# per_segment 用的是 parameters['evaluation']['segment_columns']
# test cache 沒有 cust_segment_typ → 通常為空
print("\nper_segment (compute_all_metrics):", "(empty)" if not result["per_segment"] else "")
if result["per_segment"]:
    display(pd.DataFrame(result["per_segment"]).T.sort_index())

---

通過上面所有 sanity 即代表 `metrics_spark.py` 各層在這份資料上：

- A1 `rank_within_query` ✓
- A2 `add_query_total_rel` ✓
- A3 `add_row_contributions` ✓（cum_rel / prec_at_pos / dcg_term / top_k / ap_contrib / ndcg_contrib）
- L2 `compute_per_query_metrics` ✓（每 query 一列；含 map@K / ndcg@K / precision@K / recall@K）
- B1 `aggregate_overall` ✓ 與手算一致
- B2 `aggregate_per_item` ✓ 改用 `hit_rate@K / map_attr@K / ndcg_attr@K / mean_pos`
- B3 `aggregate_per_segment` ✓ 與手算一致（per-query → 每客戶等權）
- C `macro_average` ✓
- D `compute_all_metrics` 端到端 overall 與逐步驗證 ✓

In [ ]:
spark.stop()